In [16]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
# NOTEBOOK_DIR == .../STUDY-DATA/third_week/12_24

PROJECT_ROOT = NOTEBOOK_DIR.parents[1]  # .../STUDY-DATA
DATA_DIR = PROJECT_ROOT / "third_week" / "data_csv"

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("tone_vectors exists:", (DATA_DIR / "tone_vectors.pkl").exists())
print("tone_metadata_extended exists:", (DATA_DIR / "tone_metadata_extended.csv").exists())

# 같은 폴더의 .py import 보장 (agent10_rule_engine.py, map_tone_vector_to_params.py)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

NOTEBOOK_DIR: /Users/mac/Desktop/project/STUDY-DATA/third_week/12_25
PROJECT_ROOT: /Users/mac/Desktop/project/STUDY-DATA
DATA_DIR: /Users/mac/Desktop/project/STUDY-DATA/third_week/data_csv
tone_vectors exists: True
tone_metadata_extended exists: True


Rule Engine + ToneAnalyzer

In [17]:
import numpy as np
import pandas as pd

# 네 팀원이 만든 import-safe 버전 기준:
from map_tone_vector_to_params import ToneAnalyzer

# 네가 만든 rule engine 파일이 여기에 있다고 가정:
# third_week/12_24/agent10_rule_engine.py
from agent10_rule_engine import generate_message

ToneAnalyzer 인스턴스 생성

In [18]:
PATH_CENTROIDS = DATA_DIR / "tone_vectors.pkl"
PATH_META = DATA_DIR / "tone_metadata_extended.csv"

analyzer = ToneAnalyzer.from_files(PATH_CENTROIDS, PATH_META)

print("loaded tone ids:", list(analyzer.tone_centroids.keys()))
print("meta columns:", analyzer.tone_meta.columns.tolist())

loaded tone ids: ['Scientific', 'Emotional', 'Luxury', 'Casual']
meta columns: ['keyword_count', 'keywords', 'model_used', 'dominant_trait', 'proof_level', 'emotion_level', 'cta_strength', 'lexicon_group', 'ban_group', 'description']


테스트 입력 준비 (persona / product_meta / slot_schema)

In [19]:
persona = "민감성 피부 직장인"

product_meta = {
    "name": "라네즈 워터뱅크 블루 히알루로닉 크림",
    "key_benefit": "보습 장벽 강화",
    "proof_point": "임상 데이터",
    "emotion_phrase": "편안한 일상"
}

slot_schema = ["intro", "proof", "emotion", "cta"]

tone_vector 준비 (실전용 / 더미용)
(A) 실전: tone_vector가 이미 있으면 그대로 넣기
# tone_vector = 실제 임베딩 벡터 (np.ndarray shape (D,))
# 예: tone_vector = your_vector

(B) 더미 테스트: centroid 평균으로 “그럴듯한 입력” 만들기

In [20]:
# centroid 중 하나를 골라서 약간 노이즈
tone_id_for_test = list(analyzer.tone_centroids.keys())[0]
base = analyzer.tone_centroids[tone_id_for_test]
noise = np.random.normal(0, 0.01, size=base.shape)

tone_vector = (base + noise).astype(float)

print("test tone_id base:", tone_id_for_test)
print("tone_vector shape:", tone_vector.shape)

test tone_id base: Scientific
tone_vector shape: (768,)


map_tone_vector_to_params 단독 검증

In [21]:
params = analyzer.map_tone_vector_to_params(tone_vector)
params

{'tone_id': 'Scientific',
 'dominant_trait': '논리/근거 중심',
 'proof_level': 'high',
 'emotion_level': 'low',
 'cta_strength': 'mid',
 'lexicon_group': 'scientific_words',
 'ban_group': 'emotional_words',
 'description': '근거와 데이터 기반 설명을 우선하는 톤',
 'similarity': 0.9635457014025347}

케이스 1) generate_message(persona, tone_vector, slot_schema, product_meta) 형태라면

In [22]:
result = generate_message(
    persona=persona,
    tone_vector=tone_vector,
    slot_schema=slot_schema,
    product=product_meta  
)
result

{'status': 'success',
 'tone_id': 'Scientific',
 'message': "민감성 피부 직장인님께 {'name': '라네즈 워터뱅크 블루 히알루로닉 크림', 'key_benefit': '보습 장벽 강화', 'proof_point': '임상 데이터', 'emotion_phrase': '편안한 일상'}를 소개합니다. (검증) {'name': '라네즈 워터뱅크 블루 히알루로닉 크림', 'key_benefit': '보습 장벽 강화', 'proof_point': '임상 데이터', 'emotion_phrase': '편안한 일상'}는 임상 데이터에서 검증되었습니다. (검증) 지금 바로 경험해보세요. (검증)",
 'trace': {'tone_id': 'Scientific',
  'similarity': 0.9635457014025347,
  'original_params': {'tone_id': 'Scientific',
   'dominant_trait': '논리/근거 중심',
   'proof_level': 'high',
   'emotion_level': 'low',
   'cta_strength': 'mid',
   'lexicon_group': 'scientific_words',
   'ban_group': 'emotional_words',
   'description': '근거와 데이터 기반 설명을 우선하는 톤',
   'similarity': 0.9635457014025347},
  'final_slot_order': ['intro', 'proof', 'cta']}}

케이스 2) generate_message가 analyzer/params를 외부 주입받도록 되어있다면 (예: params를 넘기는 구조)

In [23]:
# params = analyzer.map_tone_vector_to_params(tone_vector)
# result = generate_message(persona, params, slot_schema, product_meta)
# result

In [24]:
from pprint import pprint

print("\n--- MESSAGE ---")
print(result["message"])

print("\n--- TRACE ---")
pprint(result["trace"])


--- MESSAGE ---
민감성 피부 직장인님께 {'name': '라네즈 워터뱅크 블루 히알루로닉 크림', 'key_benefit': '보습 장벽 강화', 'proof_point': '임상 데이터', 'emotion_phrase': '편안한 일상'}를 소개합니다. (검증) {'name': '라네즈 워터뱅크 블루 히알루로닉 크림', 'key_benefit': '보습 장벽 강화', 'proof_point': '임상 데이터', 'emotion_phrase': '편안한 일상'}는 임상 데이터에서 검증되었습니다. (검증) 지금 바로 경험해보세요. (검증)

--- TRACE ---
{'final_slot_order': ['intro', 'proof', 'cta'],
 'original_params': {'ban_group': 'emotional_words',
                     'cta_strength': 'mid',
                     'description': '근거와 데이터 기반 설명을 우선하는 톤',
                     'dominant_trait': '논리/근거 중심',
                     'emotion_level': 'low',
                     'lexicon_group': 'scientific_words',
                     'proof_level': 'high',
                     'similarity': 0.9635457014025347,
                     'tone_id': 'Scientific'},
 'similarity': 0.9635457014025347,
 'tone_id': 'Scientific'}


trace 슬롯 단위 확인

In [25]:
trace_slots = result.get("trace", {}).get("slots", {})
trace_slots

{}

재현성 체크

In [26]:
result2 = generate_message(
    persona=persona,
    tone_vector=tone_vector,
    slot_schema=slot_schema,
    product=product_meta
)

print(result["message"] == result2["message"])
print(result["trace"] == result2["trace"])

True
True
